# FitCheck 데모 실행 (Colab)

인물 사진 + 옷 사진 → 가상 피팅 + 사이즈 추천. 합성 엔진은 FASHN VTON v1.5.

**먼저 할 것: 런타임 → 런타임 유형 변경 → GPU (T4)** 로 바꾸세요. GPU가 없으면 한 장에 수십 분 걸립니다.

셀을 위에서부터 차례로 실행하면 마지막에 **공개 링크**(`https://....gradio.live`)가 나옵니다.
그 링크를 열면 데모가 뜨고, 팀원·교수님도 같은 링크로 들어올 수 있습니다.

| | T4 (무료) | L4 · A100 (Colab 유료) |
|---|---|---|
| 한 장 합성 | 약 2분 | 더 빠름 (미측정) |

설치와 가중치 내려받기는 최초 1회 약 5분입니다. 세션을 닫으면 링크도 끊깁니다.

In [ ]:
# 1. GPU 확인
!nvidia-smi --query-gpu=name,memory.total --format=csv

In [ ]:
# 2. 코드 받기 (앱 + FASHN)
import os

APP = '/content/vfa'
FASHN = '/content/fashn-vton-1.5'
WEIGHTS = '/content/fashn-weights'

if not os.path.exists(APP):
    !git clone -q https://github.com/jhjung03eee/virtual-fitting-app.git {APP}
else:
    !git -C {APP} pull -q
if not os.path.exists(FASHN):
    !git clone -q --depth 1 https://github.com/fashn-AI/fashn-vton-1.5.git {FASHN}
print('완료')

In [ ]:
# 3. 설치 (최초 1회 약 3분)
#    mediapipe: 사진 자세·거리 검사(app/photo_check.py)에 필요. 없으면 검사를 건너뛴다
!pip install -q -e {FASHN}
!pip install -q mediapipe gradio
print('완료')

In [ ]:
# 4. 가중치 내려받기 (약 2GB, 최초 1회)
if not os.path.exists(os.path.join(WEIGHTS, 'model.safetensors')):
    !python {FASHN}/scripts/download_weights.py --weights-dir {WEIGHTS}
print('완료')

In [ ]:
# 5. 데모 실행 — 출력에 뜨는 공개 링크를 열면 됩니다 (처음 로딩 약 1분)
#    멈추려면 이 셀을 중지하세요. 링크도 같이 끊깁니다.
os.environ['FASHN_REPO'] = FASHN
os.environ['FASHN_WEIGHTS'] = WEIGHTS
!python {APP}/app/gradio_app.py

## 잘 안 될 때

- **공개 링크가 안 나옴**: 위 셀 출력 아래쪽 안내대로 `eval_js` 로 로컬 포트를 열 수 있습니다.
- **사진을 올렸는데 "다시 찍어주세요"**: 정면 · 팔은 몸에서 주먹 하나 · 머리부터 발끝까지 화면을 꽉 채워 찍은 사진이어야 합니다.
  그대로 해보려면 고급 설정의 **사진 검사 무시하고 합성**을 켜세요.
- **결과가 어색함**: 고급 설정에서 시드를 바꿔 다시 해보세요. 같은 설정도 시드에 따라 결과가 달라집니다.
- **메모리 부족**: 런타임을 다시 시작하고 5번 셀만 실행하세요 (설치·가중치는 남아 있습니다).